In [ ]:
import json
from pathlib import Path

In [ ]:
titles = {
    "degasperi.jsonl": "De Marchi's writings",
    "fiction.jsonl": "Literature",
    "moro.jsonl": "Carlo Visconti's works",
    "wikinews.jsonl": "WikiNews",
}

# ---------- plotting ----------
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for ax, fname in zip(axes, files):
    with open(base_path + fname) as f:
        data = [json.loads(line) for line in f]

    entities = flatten(extract_ner_labels(item["text"]) for item in data)
    dist = Counter(ent[1] for ent in entities)

    values = [dist.get(label, 0) for label in labels]
    colors = [label2color[label] for label in labels]

    ax.pie(
        values,
        labels=labels,
        colors=colors,
        autopct=lambda pct: f"{pct:.0f}%" if pct > 1.1 else "",
        startangle=90,
        pctdistance=0.6,
        wedgeprops={"edgecolor": "black"},
    )
    ax.set_title(titles[fname])

plt.suptitle("Distribution of NER labels by dataset", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path
import json

# results_dir = Path("../output/rener_2026-01-09_14-51-45_70b_range_0_8.json")
results_dir = Path("../output/rener_2026-02-16_12-10-11_degasperi.json")

with open(results_dir) as results_file:
    results = json.load(results_file)


float_precision = 3
for i in range(0, 8):
    avg = results[f"top_k={i}"][f"avg_metrics_topk={i}"]
    std = results[f"top_k={i}"][f"std_metrics_topk={i}"]

    print(
        rf"{i} & ${avg['precision']:.{float_precision}f} \pm {std['precision']:.{float_precision}f}$ & ${avg['recall']:.{float_precision}f} \pm {std['recall']:.{float_precision}f}$ & ${avg['f1']:.{float_precision}f} \pm {std['f1']:.{float_precision}f}$ \\"
    )


In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path


def load_metrics(filepath):
    """Load metrics from JSON file."""
    with open(filepath) as f:
        results = json.load(f)

    # Extract data
    top_k_values = []
    precision_mean, precision_std = [], []
    recall_mean, recall_std = [], []
    f1_mean, f1_std = [], []

    # Determine the range of top_k values available
    max_k = 0
    for key in results.keys():
        if key.startswith("top_k="):
            k = int(key.split("=")[1])
            max_k = max(max_k, k)

    # Extract metrics for each top_k value
    for i in range(0, max_k + 1):
        key = f"top_k={i}"
        if key in results:
            avg = results[key][f"avg_metrics_topk={i}"]
            std = results[key][f"std_metrics_topk={i}"]

            top_k_values.append(i)
            precision_mean.append(avg["precision"])
            precision_std.append(std["precision"])
            recall_mean.append(avg["recall"])
            recall_std.append(std["recall"])
            f1_mean.append(avg["f1"])
            f1_std.append(std["f1"])

    return {
        "top_k": np.array(top_k_values),
        "precision_mean": np.array(precision_mean),
        "precision_std": np.array(precision_std),
        "recall_mean": np.array(recall_mean),
        "recall_std": np.array(recall_std),
        "f1_mean": np.array(f1_mean),
        "f1_std": np.array(f1_std),
    }


# Load data for both models
results_8b_path = Path(
    "../output/rener_2026-01-09_14-51-45_70b_range_0_8.json"
)  # Update this path
results_70b_path = Path("../output/rener_2026-01-09_14-51-45_70b_range_0_8.json")

data_8b = load_metrics(results_8b_path)
data_70b = load_metrics(results_70b_path)

# Create figure with subplots
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(
    "Llama3.1 Performance Comparison: 8B vs 70B Models",
    fontsize=18,
    fontweight="bold",
    y=0.995,
)

# Color schemes
colors_8b = {"precision": "#2E86AB", "recall": "#A23B72", "f1": "#F18F01"}
colors_70b = {"precision": "#06A77D", "recall": "#D741A7", "f1": "#FFC145"}

# Row 1: 8B Model
# Precision
ax = axes[0, 0]
ax.plot(
    data_8b["top_k"],
    data_8b["precision_mean"],
    "o-",
    linewidth=2.5,
    markersize=8,
    color=colors_8b["precision"],
    label="8B",
)
ax.fill_between(
    data_8b["top_k"],
    data_8b["precision_mean"] - data_8b["precision_std"],
    data_8b["precision_mean"] + data_8b["precision_std"],
    alpha=0.25,
    color=colors_8b["precision"],
)
ax.set_xlabel("top_k", fontsize=11, fontweight="bold")
ax.set_ylabel("Precision", fontsize=11, fontweight="bold")
ax.set_title("Precision - 8B Model", fontsize=12, fontweight="bold")
ax.grid(True, alpha=0.3, linestyle="--")
ax.set_xticks(data_8b["top_k"])
ax.set_ylim([0.70, 0.95])

# Recall
ax = axes[0, 1]
ax.plot(
    data_8b["top_k"],
    data_8b["recall_mean"],
    "s-",
    linewidth=2.5,
    markersize=8,
    color=colors_8b["recall"],
    label="8B",
)
ax.fill_between(
    data_8b["top_k"],
    data_8b["recall_mean"] - data_8b["recall_std"],
    data_8b["recall_mean"] + data_8b["recall_std"],
    alpha=0.25,
    color=colors_8b["recall"],
)
ax.set_xlabel("top_k", fontsize=11, fontweight="bold")
ax.set_ylabel("Recall", fontsize=11, fontweight="bold")
ax.set_title("Recall - 8B Model", fontsize=12, fontweight="bold")
ax.grid(True, alpha=0.3, linestyle="--")
ax.set_xticks(data_8b["top_k"])
ax.set_ylim([0.70, 0.95])

# F1 Score
ax = axes[0, 2]
ax.plot(
    data_8b["top_k"],
    data_8b["f1_mean"],
    "^-",
    linewidth=2.5,
    markersize=8,
    color=colors_8b["f1"],
    label="8B",
)
ax.fill_between(
    data_8b["top_k"],
    data_8b["f1_mean"] - data_8b["f1_std"],
    data_8b["f1_mean"] + data_8b["f1_std"],
    alpha=0.25,
    color=colors_8b["f1"],
)
ax.set_xlabel("top_k", fontsize=11, fontweight="bold")
ax.set_ylabel("F1 Score", fontsize=11, fontweight="bold")
ax.set_title("F1 Score - 8B Model", fontsize=12, fontweight="bold")
ax.grid(True, alpha=0.3, linestyle="--")
ax.set_xticks(data_8b["top_k"])
ax.set_ylim([0.70, 0.95])

# Row 2: 70B Model
# Precision
ax = axes[1, 0]
ax.plot(
    data_70b["top_k"],
    data_70b["precision_mean"],
    "o-",
    linewidth=2.5,
    markersize=8,
    color=colors_70b["precision"],
    label="70B",
)
ax.fill_between(
    data_70b["top_k"],
    data_70b["precision_mean"] - data_70b["precision_std"],
    data_70b["precision_mean"] + data_70b["precision_std"],
    alpha=0.25,
    color=colors_70b["precision"],
)
ax.set_xlabel("top_k", fontsize=11, fontweight="bold")
ax.set_ylabel("Precision", fontsize=11, fontweight="bold")
ax.set_title("Precision - 70B Model", fontsize=12, fontweight="bold")
ax.grid(True, alpha=0.3, linestyle="--")
ax.set_xticks(data_70b["top_k"])
ax.set_ylim([0.70, 0.95])

# Recall
ax = axes[1, 1]
ax.plot(
    data_70b["top_k"],
    data_70b["recall_mean"],
    "s-",
    linewidth=2.5,
    markersize=8,
    color=colors_70b["recall"],
    label="70B",
)
ax.fill_between(
    data_70b["top_k"],
    data_70b["recall_mean"] - data_70b["recall_std"],
    data_70b["recall_mean"] + data_70b["recall_std"],
    alpha=0.25,
    color=colors_70b["recall"],
)
ax.set_xlabel("top_k", fontsize=11, fontweight="bold")
ax.set_ylabel("Recall", fontsize=11, fontweight="bold")
ax.set_title("Recall - 70B Model", fontsize=12, fontweight="bold")
ax.grid(True, alpha=0.3, linestyle="--")
ax.set_xticks(data_70b["top_k"])
ax.set_ylim([0.70, 0.95])

# F1 Score
ax = axes[1, 2]
ax.plot(
    data_70b["top_k"],
    data_70b["f1_mean"],
    "^-",
    linewidth=2.5,
    markersize=8,
    color=colors_70b["f1"],
    label="70B",
)
ax.fill_between(
    data_70b["top_k"],
    data_70b["f1_mean"] - data_70b["f1_std"],
    data_70b["f1_mean"] + data_70b["f1_std"],
    alpha=0.25,
    color=colors_70b["f1"],
)
ax.set_xlabel("top_k", fontsize=11, fontweight="bold")
ax.set_ylabel("F1 Score", fontsize=11, fontweight="bold")
ax.set_title("F1 Score - 70B Model", fontsize=12, fontweight="bold")
ax.grid(True, alpha=0.3, linestyle="--")
ax.set_xticks(data_70b["top_k"])
ax.set_ylim([0.70, 0.95])

plt.tight_layout()
plt.savefig("topk_metrics_8b_vs_70b_separate.png", dpi=300, bbox_inches="tight")
plt.show()

# Create combined comparison plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle(
    "Direct Comparison: 8B vs 70B Models", fontsize=18, fontweight="bold", y=1.02
)

# Precision comparison
ax = axes[0]
ax.plot(
    data_8b["top_k"],
    data_8b["precision_mean"],
    "o-",
    linewidth=2.5,
    markersize=8,
    color=colors_8b["precision"],
    label="8B",
    alpha=0.8,
)
ax.fill_between(
    data_8b["top_k"],
    data_8b["precision_mean"] - data_8b["precision_std"],
    data_8b["precision_mean"] + data_8b["precision_std"],
    alpha=0.2,
    color=colors_8b["precision"],
)
ax.plot(
    data_70b["top_k"],
    data_70b["precision_mean"],
    "s-",
    linewidth=2.5,
    markersize=8,
    color=colors_70b["precision"],
    label="70B",
    alpha=0.8,
)
ax.fill_between(
    data_70b["top_k"],
    data_70b["precision_mean"] - data_70b["precision_std"],
    data_70b["precision_mean"] + data_70b["precision_std"],
    alpha=0.2,
    color=colors_70b["precision"],
)
ax.set_xlabel("top_k", fontsize=12, fontweight="bold")
ax.set_ylabel("Precision", fontsize=12, fontweight="bold")
ax.set_title("Precision Comparison", fontsize=13, fontweight="bold")
ax.legend(fontsize=11, framealpha=0.95)
ax.grid(True, alpha=0.3, linestyle="--")
ax.set_xticks(data_8b["top_k"])
ax.set_ylim([0.70, 0.95])

# Recall comparison
ax = axes[1]
ax.plot(
    data_8b["top_k"],
    data_8b["recall_mean"],
    "o-",
    linewidth=2.5,
    markersize=8,
    color=colors_8b["recall"],
    label="8B",
    alpha=0.8,
)
ax.fill_between(
    data_8b["top_k"],
    data_8b["recall_mean"] - data_8b["recall_std"],
    data_8b["recall_mean"] + data_8b["recall_std"],
    alpha=0.2,
    color=colors_8b["recall"],
)
ax.plot(
    data_70b["top_k"],
    data_70b["recall_mean"],
    "s-",
    linewidth=2.5,
    markersize=8,
    color=colors_70b["recall"],
    label="70B",
    alpha=0.8,
)
ax.fill_between(
    data_70b["top_k"],
    data_70b["recall_mean"] - data_70b["recall_std"],
    data_70b["recall_mean"] + data_70b["recall_std"],
    alpha=0.2,
    color=colors_70b["recall"],
)
ax.set_xlabel("top_k", fontsize=12, fontweight="bold")
ax.set_ylabel("Recall", fontsize=12, fontweight="bold")
ax.set_title("Recall Comparison", fontsize=13, fontweight="bold")
ax.legend(fontsize=11, framealpha=0.95)
ax.grid(True, alpha=0.3, linestyle="--")
ax.set_xticks(data_8b["top_k"])
ax.set_ylim([0.70, 0.95])

# F1 Score comparison
ax = axes[2]
ax.plot(
    data_8b["top_k"],
    data_8b["f1_mean"],
    "o-",
    linewidth=2.5,
    markersize=8,
    color=colors_8b["f1"],
    label="8B",
    alpha=0.8,
)
ax.fill_between(
    data_8b["top_k"],
    data_8b["f1_mean"] - data_8b["f1_std"],
    data_8b["f1_mean"] + data_8b["f1_std"],
    alpha=0.2,
    color=colors_8b["f1"],
)
ax.plot(
    data_70b["top_k"],
    data_70b["f1_mean"],
    "s-",
    linewidth=2.5,
    markersize=8,
    color=colors_70b["f1"],
    label="70B",
    alpha=0.8,
)
ax.fill_between(
    data_70b["top_k"],
    data_70b["f1_mean"] - data_70b["f1_std"],
    data_70b["f1_mean"] + data_70b["f1_std"],
    alpha=0.2,
    color=colors_70b["f1"],
)
ax.set_xlabel("top_k", fontsize=12, fontweight="bold")
ax.set_ylabel("F1 Score", fontsize=12, fontweight="bold")
ax.set_title("F1 Score Comparison", fontsize=13, fontweight="bold")
ax.legend(fontsize=11, framealpha=0.95)
ax.grid(True, alpha=0.3, linestyle="--")
ax.set_xticks(data_8b["top_k"])
ax.set_ylim([0.70, 0.95])

plt.tight_layout()
plt.savefig("topk_metrics_8b_vs_70b_combined.png", dpi=300, bbox_inches="tight")
plt.show()

# Print summary statistics
print("\n" + "=" * 80)
print("SUMMARY STATISTICS - 8B MODEL")
print("=" * 80)
print(
    f"Best Precision: {data_8b['precision_mean'].max():.3f} ± {data_8b['precision_std'][data_8b['precision_mean'].argmax()]:.3f} at top_k={data_8b['top_k'][data_8b['precision_mean'].argmax()]}"
)
print(
    f"Best Recall:    {data_8b['recall_mean'].max():.3f} ± {data_8b['recall_std'][data_8b['recall_mean'].argmax()]:.3f} at top_k={data_8b['top_k'][data_8b['recall_mean'].argmax()]}"
)
print(
    f"Best F1 Score:  {data_8b['f1_mean'].max():.3f} ± {data_8b['f1_std'][data_8b['f1_mean'].argmax()]:.3f} at top_k={data_8b['top_k'][data_8b['f1_mean'].argmax()]}"
)

print("\n" + "=" * 80)
print("SUMMARY STATISTICS - 70B MODEL")
print("=" * 80)
print(
    f"Best Precision: {data_70b['precision_mean'].max():.3f} ± {data_70b['precision_std'][data_70b['precision_mean'].argmax()]:.3f} at top_k={data_70b['top_k'][data_70b['precision_mean'].argmax()]}"
)
print(
    f"Best Recall:    {data_70b['recall_mean'].max():.3f} ± {data_70b['recall_std'][data_70b['recall_mean'].argmax()]:.3f} at top_k={data_70b['top_k'][data_70b['recall_mean'].argmax()]}"
)
print(
    f"Best F1 Score:  {data_70b['f1_mean'].max():.3f} ± {data_70b['f1_std'][data_70b['f1_mean'].argmax()]:.3f} at top_k={data_70b['top_k'][data_70b['f1_mean'].argmax()]}"
)
print("=" * 80)

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# Load results from JSON file
results_dir = Path("../output/rener_2026-01-09_14-51-45_70b_range_0_8.json")
with open(results_dir) as results_file:
    results = json.load(results_file)

# Extract data
top_k_values = []
precision_mean, precision_std = [], []
recall_mean, recall_std = [], []
f1_mean, f1_std = [], []

# Determine the range of top_k values available
max_k = 0
for key in results.keys():
    if key.startswith("top_k="):
        k = int(key.split("=")[1])
        max_k = max(max_k, k)

# Extract metrics for each top_k value
for i in range(0, max_k + 1):
    key = f"top_k={i}"
    if key in results:
        avg = results[key][f"avg_metrics_topk={i}"]
        std = results[key][f"std_metrics_topk={i}"]

        top_k_values.append(i)
        precision_mean.append(avg["precision"])
        precision_std.append(std["precision"])
        recall_mean.append(avg["recall"])
        recall_std.append(std["recall"])
        f1_mean.append(avg["f1"])
        f1_std.append(std["f1"])

# Convert to numpy arrays
top_k = np.array(top_k_values)
precision_mean = np.array(precision_mean)
precision_std = np.array(precision_std)
recall_mean = np.array(recall_mean)
recall_std = np.array(recall_std)
f1_mean = np.array(f1_mean)
f1_std = np.array(f1_std)

# Create visualization with shaded error regions
fig, ax = plt.subplots(figsize=(12, 8))

# Plot lines with shaded standard deviation bands
ax.plot(
    top_k,
    precision_mean,
    "o-",
    label="Precision",
    linewidth=2.5,
    markersize=9,
    color="#2E86AB",
)
ax.fill_between(
    top_k,
    precision_mean - precision_std,
    precision_mean + precision_std,
    alpha=0.25,
    color="#2E86AB",
)

ax.plot(
    top_k,
    recall_mean,
    "s-",
    label="Recall",
    linewidth=2.5,
    markersize=9,
    color="#A23B72",
)
ax.fill_between(
    top_k,
    recall_mean - recall_std,
    recall_mean + recall_std,
    alpha=0.25,
    color="#A23B72",
)

ax.plot(
    top_k, f1_mean, "^-", label="F1 Score", linewidth=2.5, markersize=9, color="#F18F01"
)
ax.fill_between(top_k, f1_mean - f1_std, f1_mean + f1_std, alpha=0.25, color="#F18F01")

# Formatting
ax.set_xlabel("top_k", fontsize=14, fontweight="bold")
ax.set_ylabel("Score", fontsize=14, fontweight="bold")
ax.set_title(
    "Llama3.1 8B Performance Metrics vs top_k (±1 std)",
    fontsize=16,
    fontweight="bold",
    pad=20,
)
ax.legend(
    loc="lower right",
    framealpha=0.95,
    fontsize=12,
    frameon=True,
    shadow=True,
    fancybox=True,
)
ax.grid(True, alpha=0.3, linestyle="--", linewidth=0.8)
ax.set_xticks(top_k)
ax.set_ylim([0.70, 0.95])

# Add minor gridlines
ax.minorticks_on()
ax.grid(which="minor", alpha=0.15, linestyle=":", linewidth=0.5)

plt.tight_layout()
plt.savefig("topk_metrics_std_bands.png", dpi=300, bbox_inches="tight")
plt.show()

# Print summary
print("\n" + "=" * 70)
print("SUMMARY STATISTICS")
print("=" * 70)
print(
    f"\nBest Precision: {precision_mean.max():.3f} ± {precision_std[precision_mean.argmax()]:.3f} at top_k={top_k[precision_mean.argmax()]}"
)
print(
    f"Best Recall:    {recall_mean.max():.3f} ± {recall_std[recall_mean.argmax()]:.3f} at top_k={top_k[recall_mean.argmax()]}"
)
print(
    f"Best F1 Score:  {f1_mean.max():.3f} ± {f1_std[f1_mean.argmax()]:.3f} at top_k={top_k[f1_mean.argmax()]}"
)
print("=" * 70)

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path


def load_metrics(filepath):
    """Load metrics from JSON file."""
    with open(filepath) as f:
        results = json.load(f)

    # Extract data
    top_k_values = []
    precision_mean, precision_std = [], []
    recall_mean, recall_std = [], []
    f1_mean, f1_std = [], []

    # Determine the range of top_k values available
    max_k = 0
    for key in results.keys():
        if key.startswith("top_k="):
            k = int(key.split("=")[1])
            max_k = max(max_k, k)

    # Extract metrics for each top_k value
    for i in range(0, max_k + 1):
        key = f"top_k={i}"
        if key in results:
            avg = results[key][f"avg_metrics_topk={i}"]
            std = results[key][f"std_metrics_topk={i}"]

            top_k_values.append(i)
            precision_mean.append(avg["precision"])
            precision_std.append(std["precision"])
            recall_mean.append(avg["recall"])
            recall_std.append(std["recall"])
            f1_mean.append(avg["f1"])
            f1_std.append(std["f1"])

    return {
        "top_k": np.array(top_k_values),
        "precision_mean": np.array(precision_mean),
        "precision_std": np.array(precision_std),
        "recall_mean": np.array(recall_mean),
        "recall_std": np.array(recall_std),
        "f1_mean": np.array(f1_mean),
        "f1_std": np.array(f1_std),
    }


# Load data for both models
results_8b_path = Path(
    "../output/rener_2026-01-09_10-47-21_8b_range_0_15.json"
)  # Update this path
results_70b_path = Path("../output/rener_2026-01-09_14-51-45_70b_range_0_8.json")

data_8b = load_metrics(results_8b_path)
data_70b = load_metrics(results_70b_path)

# Create figure with bar charts
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle(
    "Performance Metrics Comparison: 8B vs 70B Models",
    fontsize=18,
    fontweight="bold",
    y=0.995,
)

# Colors
colors = ["#2E86AB", "#A23B72", "#F18F01"]

# 8B Model Bar Chart
ax = axes[0]
x = np.arange(len(data_8b["top_k"]))
width = 0.25

bars1 = ax.bar(
    x - width,
    data_8b["precision_mean"],
    width,
    yerr=data_8b["precision_std"],
    label="Precision",
    color=colors[0],
    alpha=0.8,
    capsize=4,
    edgecolor="black",
    linewidth=0.7,
)
bars2 = ax.bar(
    x,
    data_8b["recall_mean"],
    width,
    yerr=data_8b["recall_std"],
    label="Recall",
    color=colors[1],
    alpha=0.8,
    capsize=4,
    edgecolor="black",
    linewidth=0.7,
)
bars3 = ax.bar(
    x + width,
    data_8b["f1_mean"],
    width,
    yerr=data_8b["f1_std"],
    label="F1 Score",
    color=colors[2],
    alpha=0.8,
    capsize=4,
    edgecolor="black",
    linewidth=0.7,
)

ax.set_xlabel("top_k", fontsize=13, fontweight="bold")
ax.set_ylabel("Score", fontsize=13, fontweight="bold")
ax.set_title(
    "8B Model - Metrics by top_k (with error bars)", fontsize=14, fontweight="bold"
)
ax.set_xticks(x)
ax.set_xticklabels(data_8b["top_k"])
ax.legend(fontsize=11, framealpha=0.95, loc="lower right")
ax.grid(True, alpha=0.3, axis="y", linestyle="--")
ax.set_ylim([0.70, 0.95])

# Add value labels on bars
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2.0,
            height,
            f"{height:.3f}",
            ha="center",
            va="bottom",
            fontsize=7,
            rotation=0,
        )

# 70B Model Bar Chart
ax = axes[1]
x = np.arange(len(data_70b["top_k"]))

bars1 = ax.bar(
    x - width,
    data_70b["precision_mean"],
    width,
    yerr=data_70b["precision_std"],
    label="Precision",
    color=colors[0],
    alpha=0.8,
    capsize=4,
    edgecolor="black",
    linewidth=0.7,
)
bars2 = ax.bar(
    x,
    data_70b["recall_mean"],
    width,
    yerr=data_70b["recall_std"],
    label="Recall",
    color=colors[1],
    alpha=0.8,
    capsize=4,
    edgecolor="black",
    linewidth=0.7,
)
bars3 = ax.bar(
    x + width,
    data_70b["f1_mean"],
    width,
    yerr=data_70b["f1_std"],
    label="F1 Score",
    color=colors[2],
    alpha=0.8,
    capsize=4,
    edgecolor="black",
    linewidth=0.7,
)

ax.set_xlabel("top_k", fontsize=13, fontweight="bold")
ax.set_ylabel("Score", fontsize=13, fontweight="bold")
ax.set_title(
    "70B Model - Metrics by top_k (with error bars)", fontsize=14, fontweight="bold"
)
ax.set_xticks(x)
ax.set_xticklabels(data_70b["top_k"])
ax.legend(fontsize=11, framealpha=0.95, loc="lower right")
ax.grid(True, alpha=0.3, axis="y", linestyle="--")
ax.set_ylim([0.70, 0.95])

# Add value labels on bars
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2.0,
            height,
            f"{height:.3f}",
            ha="center",
            va="bottom",
            fontsize=7,
            rotation=0,
        )

plt.tight_layout()
plt.savefig("topk_metrics_bar_chart.png", dpi=300, bbox_inches="tight")
plt.show()

# Create side-by-side comparison for each top_k
num_topk = len(data_8b["top_k"])
fig, axes = plt.subplots(3, 3, figsize=(18, 12))
fig.suptitle(
    "Detailed Comparison by top_k Value", fontsize=18, fontweight="bold", y=0.995
)

for idx, k in enumerate(data_8b["top_k"]):
    row = idx // 3
    col = idx % 3
    ax = axes[row, col]

    x = np.arange(3)
    width = 0.35

    metrics_8b = [
        data_8b["precision_mean"][idx],
        data_8b["recall_mean"][idx],
        data_8b["f1_mean"][idx],
    ]
    stds_8b = [
        data_8b["precision_std"][idx],
        data_8b["recall_std"][idx],
        data_8b["f1_std"][idx],
    ]

    metrics_70b = [
        data_70b["precision_mean"][idx],
        data_70b["recall_mean"][idx],
        data_70b["f1_mean"][idx],
    ]
    stds_70b = [
        data_70b["precision_std"][idx],
        data_70b["recall_std"][idx],
        data_70b["f1_std"][idx],
    ]

    bars1 = ax.bar(
        x - width / 2,
        metrics_8b,
        width,
        yerr=stds_8b,
        label="8B",
        color="#3498db",
        alpha=0.8,
        capsize=4,
        edgecolor="black",
        linewidth=0.7,
    )
    bars2 = ax.bar(
        x + width / 2,
        metrics_70b,
        width,
        yerr=stds_70b,
        label="70B",
        color="#e74c3c",
        alpha=0.8,
        capsize=4,
        edgecolor="black",
        linewidth=0.7,
    )

    ax.set_ylabel("Score", fontsize=10, fontweight="bold")
    ax.set_title(f"top_k = {k}", fontsize=11, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(["Precision", "Recall", "F1"], fontsize=9)
    ax.legend(fontsize=9, framealpha=0.95)
    ax.grid(True, alpha=0.3, axis="y", linestyle="--")
    ax.set_ylim([0.70, 0.95])

    # Add value labels
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            ax.text(
                bar.get_x() + bar.get_width() / 2.0,
                height,
                f"{height:.3f}",
                ha="center",
                va="bottom",
                fontsize=7,
            )

plt.tight_layout()
plt.savefig("topk_metrics_detailed_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

print("\n" + "=" * 80)
print("Bar charts generated successfully!")
print("=" * 80)
print("Files saved:")
print("  1. topk_metrics_bar_chart.png - Main bar chart comparison")
print("  2. topk_metrics_detailed_comparison.png - Detailed comparison by top_k")
print("=" * 80)

In [ ]:
import pandas as pd

ds_dir = "../data/kind/parallel/fiction.jsonl"
with open(ds_dir) as ds:
    df = pd.read_json(ds, lines=True)


mask = df["text"].str.contains(r"\(\(|\[\[|\{\{", regex=True)
filtered_df = df[mask]

out_dir = "../data/kind/parallel/fiction_filtered.jsonl"
filtered_df.to_json(
    out_dir,
    lines=True,
    orient="records",
)

In [ ]:
import os


def filter_and_convert_folder(input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    for filename in os.listdir(input_dir):
        if not filename.endswith(".tsv"):
            continue

        input_path = os.path.join(input_dir, filename)
        output_path = os.path.join(output_dir, filename)

        with open(input_path, "r", encoding="utf-8") as f:
            content = f.read().strip()

        sentences = content.split("\n\n")
        output_sentences = []

        for sentence in sentences:
            lines = sentence.split("\n")
            tokens, tags = [], []

            for line in lines:
                if "\t" in line:
                    tok, tag = line.split("\t")
                    tokens.append(tok)
                    tags.append(tag)

            if all(tag == "O" for tag in tags):
                continue

            bio_tags = []
            prev_tag = "O"

            for tag in tags:
                if tag == "O":
                    bio_tags.append("O")
                else:
                    if prev_tag != tag:
                        bio_tags.append(f"B-{tag}")
                    else:
                        bio_tags.append(f"I-{tag}")
                prev_tag = tag

            output_sentences.append(
                "\n".join(f"{t}\t{b}" for t, b in zip(tokens, bio_tags))
            )

        with open(output_path, "w", encoding="utf-8") as f:
            f.write("\n\n".join(output_sentences))


filter_and_convert_folder("../data/kind", "../data/kind/filtered")

In [ ]:
import json


def conll_to_jsonl(input_tsv, output_jsonl):
    with (
        open(input_tsv, "r", encoding="utf-8") as f_in,
        open(output_jsonl, "w", encoding="utf-8") as f_out,
    ):
        tokens, tags = [], []
        sent_id = 0

        for line in f_in:
            line = line.strip()
            if not line:
                if tokens:
                    example = {"id": f"{sent_id}", "tokens": tokens, "ner_tags": tags}
                    f_out.write(json.dumps(example) + "\n")
                    sent_id += 1
                    tokens, tags = [], []
                continue

            if "\t" not in line:
                continue
            token, tag = line.split("\t")
            tokens.append(token)
            tags.append(tag)

        # last sentence
        if tokens:
            example = {"id": f"{sent_id}", "tokens": tokens, "ner_tags": tags}
            f_out.write(json.dumps(example) + "\n")


conll_to_jsonl(
    "../data/kind/filtered/wikinews_train.tsv",
    "../data/kind/filtered/wikinews_train.jsonl",
)
conll_to_jsonl(
    "../data/kind/filtered/wikinews_test.tsv",
    "../data/kind/filtered/wikinews_test.jsonl",
)